In [1]:
import pandas as pd
import json

In [2]:
txt_df = pd.read_csv("alphabetical_index.csv", dtype = str)
scraped_df = pd.read_csv("patent_test_text.csv", dtype = str)
txt_df = txt_df.reset_index(drop = True)
scraped_df = scraped_df.reset_index(drop = True)
headers = scraped_df["Index"].astype(str) == "Index"
header_lst = scraped_df.index[headers].tolist()

In [ ]:
grps = []

if header_lst[0] > 0:
    first_grp = scraped_df.iloc[:header_lst[0]]
    grps.append(first_grp.to_dict("records"))
else:
    grps.append([])

for i in range(len(header_lst)):
    start = header_lst[i] + 1
    end = header_lst[i + 1] if i + 1 < len(header_lst) else len(scraped_df)
    section = scraped_df.iloc[start:end]
    section = section[section["Index"] != "Index"]
    grps.append(section.to_dict("records") if not section.empty else [])

if len(txt_df) > len(grps):
    grps.extend([[]] * (len(txt_df) - len(grps)))

sections = grps[:len(txt_df)]


In [4]:
txt_df["annotations_json"] = pd.Series(grps).apply(json.dumps)

In [ ]:
txt_df.to_csv("reformatted_02172026.csv", index = False)

In [103]:
len(txt_df)

32763

In [ ]:
(scraped_df["Index"] != "Index").sum()

43913

In [9]:
(scraped_df["Index"] == "Index").sum()

35223

In [102]:
len(grps)

35224

In [6]:
import pandas as pd
import json

# Read raw lines
with open("patent_test_text.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()

header = "Index,Verbatim,Name,Start,End,OddsLog10,Cardinality,AnnotNomenType,WordsBefore,WordsAfter"

groups = []
current_block = []
group_id = -1

for line in lines:
    line = line.strip()
    
    if line == header:
        # Save previous block if it exists
        if current_block:
            groups.append((group_id, current_block))
            current_block = []
        group_id += 1
    else:
        if line:  # skip empty lines
            current_block.append(line)

# Add last block
if current_block:
    groups.append((group_id, current_block))

# Convert each block to JSON
rows = []

for group_id, block in groups:
    # Convert block lines into a DataFrame
    df_block = pd.read_csv(
        pd.io.common.StringIO(header + "\n" + "\n".join(block))
    )
    
    # Convert to JSON (list of dicts)
    annotations_json = df_block.to_dict(orient="records")
    
    rows.append({
        "Index": group_id,
        "annotations_json": json.dumps(annotations_json)
    })

# Final dataframe
final_df = pd.DataFrame(rows)

print(final_df)

       Index                                   annotations_json
0         -1  [{"Index": "\ufeffIndex", "Verbatim": "Verbati...
1          0  [{"Index": 0, "Verbatim": NaN, "Name": "Idaeob...
2          1  [{"Index": 0, "Verbatim": NaN, "Name": "Neoreg...
3          2  [{"Index": 0, "Verbatim": NaN, "Name": "Altha"...
4          4  [{"Index": 0, "Verbatim": NaN, "Name": "Plectr...
...      ...                                                ...
25944  35217  [{"Index": 0, "Verbatim": NaN, "Name": "Lobeli...
25945  35218  [{"Index": 0, "Verbatim": NaN, "Name": "Rhodod...
25946  35219  [{"Index": 0, "Verbatim": NaN, "Name": "Vitis ...
25947  35221  [{"Index": 0, "Verbatim": NaN, "Name": "Rosa h...
25948  35222  [{"Index": 0, "Verbatim": NaN, "Name": "P. int...

[25949 rows x 2 columns]


In [ ]:
final_df.to_csv("reformatted_02172026.csv", index = False)